<h1 align="center">ДЗ 2.2: Оптимизация вычислений в Spark</h1>

# Задание 3: Пользовательские признаки

## Запускаем спарк сессию

In [14]:
from pyspark.sql import SparkSession

In [15]:
spark = (
    SparkSession.builder
    .appName("hw22-user-features")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "optimization-jupyter")
    .config("spark.executor.memory", "3g")
    .config("spark.executor.cores", "4")
    .config("spark.cores.max", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

## Подключаемся к S3 (RustFS)

In [16]:
# Параметры подключения к RustFS
S3_ENDPOINT = "http://rustfs:9000"
S3_ACCESS_KEY = "scd2admin"
S3_SECRET_KEY = "scd2-secret-2026"
S3_BUCKET = "spark-optimization"

# Получаем конфигурацию Hadoop
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

# Настройки S3A
hadoop_conf.set("fs.s3a.endpoint", S3_ENDPOINT)
hadoop_conf.set("fs.s3a.access.key", S3_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", S3_SECRET_KEY)
hadoop_conf.set(
    "fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
print("Конфигурация S3A установлена")

Конфигурация S3A установлена


In [17]:
from py4j.java_gateway import java_import

In [18]:
java_import(spark.sparkContext._jvm, "org.apache.hadoop.fs.Path")
path = spark.sparkContext._jvm.Path(f"s3a://{S3_BUCKET}/")
fs = path.getFileSystem(hadoop_conf)

## Загрузим данные в S3

Длительность просмотра контента пользователями:

In [19]:
TRANSACTIONS_LOCAL_PATH = "/home/jupyter/data/task3/cut_transactions.csv"
TRANSACTIONS_S3_PATH = f"s3a://{S3_BUCKET}/source/transactions/cut_transactions.csv"

transactions_local_path = spark.sparkContext._jvm.Path(f"file:{TRANSACTIONS_LOCAL_PATH}")
transactions_target_path = spark.sparkContext._jvm.Path(TRANSACTIONS_S3_PATH)

if fs.exists(transactions_target_path):
    print("Файл с транзакциями уже загружен")
else:
    fs.copyFromLocalFile(False, True, transactions_local_path, transactions_target_path)
    print("Файл с транзакциями загружен")

Файл с транзакциями загружен


Оценки, поставленные пользователями фильмам:

In [20]:
RATINGS_LOCAL_PATH = "/home/jupyter/data/task3/cut_ratings.csv"
RATINGS_S3_PATH = f"s3a://{S3_BUCKET}/source/ratings/cut_ratings.csv"

ratings_local_path = spark.sparkContext._jvm.Path(f"file:{RATINGS_LOCAL_PATH}")
ratings_target_path = spark.sparkContext._jvm.Path(RATINGS_S3_PATH)

if fs.exists(ratings_target_path):
    print("Файл с оценками уже загружен")
else:
    fs.copyFromLocalFile(False, True, ratings_local_path, ratings_target_path)
    print("Файл с оценками загружен")

Файл с оценками загружен


Выборка пользователей, для которых нужно рассчитать признаки:

In [21]:
IDS_LOCAL_PATH = "/home/jupyter/data/task3/ids.csv"
IDS_S3_PATH = f"s3a://{S3_BUCKET}/source/ids/ids.csv"

ids_local_path = spark.sparkContext._jvm.Path(f"file:{IDS_LOCAL_PATH}")
ids_target_path = spark.sparkContext._jvm.Path(IDS_S3_PATH)

if fs.exists(ids_target_path):
    print("Файл с пользователями уже загружен")
else:
    fs.copyFromLocalFile(False, True, ids_local_path, ids_target_path)
    print("Файл с пользователями загружен")

Файл с пользователями загружен


## Читаем из S3

In [22]:
transactions_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(TRANSACTIONS_S3_PATH)
)

In [26]:
transactions_df.printSchema()

root
 |-- element_uid: integer (nullable = true)
 |-- user_uid: integer (nullable = true)
 |-- watched_time: integer (nullable = true)



In [27]:
transactions_df.count()

274385

In [28]:
transactions_df.show(5)

+-----------+--------+------------+
|element_uid|user_uid|watched_time|
+-----------+--------+------------+
|       6130|  563180|        3264|
|       2771|  133315|       10557|
|        984|  290104|        6845|
|       6980|  385385|        2734|
|       2714|  408794|        7583|
+-----------+--------+------------+
only showing top 5 rows



In [30]:
ratings_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RATINGS_S3_PATH)
)

In [31]:
ratings_df.printSchema()

root
 |-- user_uid: integer (nullable = true)
 |-- element_uid: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- ts: double (nullable = true)



In [33]:
ratings_df.count()

438790

In [34]:
ratings_df.show(5)

+--------+-----------+------+--------------------+
|user_uid|element_uid|rating|                  ts|
+--------+-----------+------+--------------------+
|  571252|       1364|    10| 4.430517426309871E7|
|   63140|       3037|    10|4.4305139282818206E7|
|  443817|       4363|     8| 4.430513620584908E7|
|  359870|       1364|    10| 4.430506300637321E7|
|  359870|       3578|     9| 4.430506073913281E7|
+--------+-----------+------+--------------------+
only showing top 5 rows



In [35]:
users_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(IDS_S3_PATH)
)

In [36]:
users_df.printSchema()

root
 |-- user_uid: integer (nullable = true)



In [37]:
users_df.count()

100

In [38]:
users_df.show(5)

+--------+
|user_uid|
+--------+
|  110138|
|  412991|
|   83691|
|  334052|
|   24124|
+--------+
only showing top 5 rows



## Проверка ключей и дубликатов

Проверим, что в ключах нет NULL:

In [47]:
transactions_df.filter(F.col("user_uid").isNull() | F.col("element_uid").isNull()).count()

0

In [46]:
ratings_df.filter(F.col("user_uid").isNull() | F.col("element_uid").isNull()).count()

0

In [49]:
users_df.filter(F.col("user_uid").isNull()).count()

0

### Дубликаты по ключам

Просмотры:

In [50]:
transactions_duplicates = transactions_df.groupBy("user_uid", "element_uid").count().filter(F.col("count") > 1)

In [51]:
transactions_duplicates.count()

0

Оценки:

In [52]:
ratings_duplicates = ratings_df.groupBy("user_uid", "element_uid").count().filter(F.col("count") > 1)

In [53]:
ratings_duplicates.count()

0

In [54]:
users_duplicates = users_df.groupBy("user_uid").count().filter(F.col("count") > 1)

In [55]:
users_duplicates.count()

0

### Вывод

Повторов нет, дубликатов по ключам тоже, след-но случайное размножение строк при join исключено

## 1: Для каждого пользователя из выборки рассчитайте минимальную и максимальную длительность просмотра отдельно для фильмов с оценками 8, 9 и 10. Получится шесть признаков: по два для каждой оценки

### Фильтрация

В ratings_df ставим только записи с оценками 8, 9 и 10, чтобы не обрабатывать ненужные данные:

In [40]:
from pyspark.sql import functions as F

In [41]:
filtered_ratings = ratings_df.filter(F.col("rating").isin(8, 9, 10))

Отфильтруем просмотры и оценки только для пользователей из выборки, чтобы не делать джоины для ненужных пользователей:

In [58]:
transactions_filtered = transactions_df.join(users_df, ["user_uid"], "left_semi")
ratings_filtered = filtered_ratings.join(users_df, ["user_uid"], "left_semi")

Используем left_semi join, тк нужно оставить только те записи, для которых пользователь присутствует в выборке

### JOIN

Соединяем просмотры с оценками по (user_uid, element_uid):

In [59]:
views_ratings = transactions_filtered.join(ratings_filtered, ["user_uid", "element_uid"], "inner")

In [60]:
views_ratings.show(5)

+--------+-----------+------------+------+--------------------+
|user_uid|element_uid|watched_time|rating|                  ts|
+--------+-----------+------------+------+--------------------+
|   78792|        994|        5889|    10| 4.429784411292965E7|
|   78792|       7371|        6651|    10| 4.426638080812672E7|
|  129810|       9108|        7075|    10| 4.424508866766371E7|
|  409451|       8546|        4864|     8|4.4231929280472465E7|
|  193809|       9898|        5491|     8| 4.421133075362707E7|
+--------+-----------+------------+------+--------------------+
only showing top 5 rows



In [61]:
views_ratings.count()

150

### Агрегация

Признаки называем так: 

`feat_{название_агрегирующей_функции}_watched_time_rating_{оценка}`

In [63]:
features_agg = (
    views_ratings
    .groupBy("user_uid")
    .agg(
        F.min(F.when(F.col("rating") == 8, F.col("watched_time")))
         .alias("feat_min_watched_time_rating_8"),

        F.max(F.when(F.col("rating") == 8, F.col("watched_time")))
         .alias("feat_max_watched_time_rating_8"),

        F.min(F.when(F.col("rating") == 9, F.col("watched_time")))
         .alias("feat_min_watched_time_rating_9"),

        F.max(F.when(F.col("rating") == 9, F.col("watched_time")))
         .alias("feat_max_watched_time_rating_9"),

        F.min(F.when(F.col("rating") == 10, F.col("watched_time")))
         .alias("feat_min_watched_time_rating_10"),

        F.max(F.when(F.col("rating") == 10, F.col("watched_time")))
         .alias("feat_max_watched_time_rating_10")
    )
)

### Сохранение всех юзеров

Сохраните в результате всех пользователей из выборки: по одной строке на уникального
пользователя. Если для определённой оценки нет подходящих просмотров, оба признака этой
оценки должны быть NULL. При отсутствии всех трёх оценок все шесть признаков равны NULL; не
заменяйте их нулями.

Для этого присоединяем features_agg к users_df через left JOIN:

In [64]:
user_features = users_df.join(features_agg, "user_uid", "left")

In [72]:
user_features.show(vertical=True)

-RECORD 0---------------------------------
 user_uid                        | 412991 
 feat_min_watched_time_rating_8  | NULL   
 feat_max_watched_time_rating_8  | NULL   
 feat_min_watched_time_rating_9  | NULL   
 feat_max_watched_time_rating_9  | NULL   
 feat_min_watched_time_rating_10 | NULL   
 feat_max_watched_time_rating_10 | NULL   
-RECORD 1---------------------------------
 user_uid                        | 453355 
 feat_min_watched_time_rating_8  | NULL   
 feat_max_watched_time_rating_8  | NULL   
 feat_min_watched_time_rating_9  | NULL   
 feat_max_watched_time_rating_9  | NULL   
 feat_min_watched_time_rating_10 | NULL   
 feat_max_watched_time_rating_10 | NULL   
-RECORD 2---------------------------------
 user_uid                        | 334052 
 feat_min_watched_time_rating_8  | NULL   
 feat_max_watched_time_rating_8  | NULL   
 feat_min_watched_time_rating_9  | NULL   
 feat_max_watched_time_rating_9  | NULL   
 feat_min_watched_time_rating_10 | NULL   
 feat_max_w

Итоговая таблица признаков:

In [73]:
user_features.toPandas()

,user_uid,feat_min_watched_time_rating_8,feat_max_watched_time_rating_8,feat_min_watched_time_rating_9,feat_max_watched_time_rating_9,feat_min_watched_time_rating_10,feat_max_watched_time_rating_10
0,478224,5863.0,6174.0,6181.0,6181.0,NaN,NaN
1,528372,NaN,NaN,NaN,NaN,NaN,NaN
2,549225,NaN,NaN,NaN,NaN,11389.0,13999.0
3,509014,NaN,NaN,NaN,NaN,NaN,NaN
4,98390,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
95,466361,NaN,NaN,NaN,NaN,NaN,NaN
96,129810,5599.0,7129.0,NaN,NaN,7075.0,7075.0
97,412617,27645.0,27645.0,NaN,NaN,8378.0,8378.0
98,101205,6930.0,7661.0,NaN,NaN,NaN,NaN


## Проверка

Проверим уникальность пользователей и сохранность всей исходной выборки:

In [74]:
print("Количество пользователей в исходной выборке:", users_df.count())
print("Количество строк в итоговой таблице:", user_features.count())
print("Количество уникальных пользователей:", user_features.select("user_uid").distinct().count())

Количество пользователей в исходной выборке: 100
Количество строк в итоговой таблице: 100
Количество уникальных пользователей: 100


Если для определённой оценки нет подходящих просмотров, оба признака этой
оценки должны быть NULL:

In [76]:
null_8 = user_features.filter(
    (F.col("feat_min_watched_time_rating_8").isNull() !=
     F.col("feat_max_watched_time_rating_8").isNull())
)
null_9 = user_features.filter(
    (F.col("feat_min_watched_time_rating_9").isNull() !=
     F.col("feat_max_watched_time_rating_9").isNull())
)
null_10 = user_features.filter(
    (F.col("feat_min_watched_time_rating_10").isNull() !=
     F.col("feat_max_watched_time_rating_10").isNull())
)
print("Некорректных пар для оценки 8:", null_8.count())
print("Некорректных пар для оценки 9:", null_9.count())
print("Некорректных пар для оценки 10:", null_10.count())

Некорректных пар для оценки 8: 0
Некорректных пар для оценки 9: 0
Некорректных пар для оценки 10: 0


Для непустой пары признаков минимум не должен превышать максимум:

In [77]:
minmax_8 = user_features.filter(
    F.col("feat_min_watched_time_rating_8") > F.col("feat_max_watched_time_rating_8")
)
minmax_9 = user_features.filter(
    F.col("feat_min_watched_time_rating_9") > F.col("feat_max_watched_time_rating_9")
)
minmax_10 = user_features.filter(
    F.col("feat_min_watched_time_rating_10") > F.col("feat_max_watched_time_rating_10")
)
print("Некорректных пар для оценки 8:", minmax_8.count())
print("Некорректных пар для оценки 9:", minmax_9.count())
print("Некорректных пар для оценки 10:", minmax_10.count())

Некорректных пар для оценки 8: 0
Некорректных пар для оценки 9: 0
Некорректных пар для оценки 10: 0


Проверим пользователя, у которого есть все 3 оценки

Для этого сначала посчитаем, сколько разных оценок встречается у каждого пользователя:

In [82]:
user_rating_counts = (
    views_ratings
    .groupBy("user_uid")
    .agg(F.countDistinct("rating").alias("ratings_count"))
)

user_rating_counts.show()

+--------+-------------+
|user_uid|ratings_count|
+--------+-------------+
|  478224|            2|
|  549225|            1|
|  183700|            3|
|  216377|            1|
|  189082|            1|
|  187292|            1|
|  588513|            1|
|   97634|            1|
|  570263|            2|
|   74940|            2|
|  421490|            2|
|  414478|            1|
|   55511|            1|
|  353458|            1|
|  158804|            1|
|  324599|            1|
|   82440|            1|
|   28855|            2|
|  262136|            2|
|  247177|            1|
+--------+-------------+
only showing top 20 rows



Выберем пользователя, у которого есть все три оценки:

In [84]:
example_user = (
    user_rating_counts
    .filter(F.col("ratings_count") == 3)
    .select("user_uid")
    .first()["user_uid"]
)

print("user:", example_user)

user: 183700


Посмотрим его исходные просмотры и оценки:

In [85]:
views_ratings.filter(F.col("user_uid") == example_user).orderBy("rating", "watched_time").show()

+--------+-----------+------------+------+--------------------+
|user_uid|element_uid|watched_time|rating|                  ts|
+--------+-----------+------------+------+--------------------+
|  183700|       7423|        4602|     8|4.3743757279452644E7|
|  183700|       2714|        7681|     9|4.2515759309410326E7|
|  183700|        747|        7595|    10|  4.23412583987059E7|
+--------+-----------+------------+------+--------------------+



То есть у него на каждую оценку по одному просмотру, следовательно min и max для каждой оценки будут совпадать

Его итоговые признаки:

In [87]:
user_features.filter(F.col("user_uid") == example_user).toPandas()

,user_uid,feat_min_watched_time_rating_8,feat_max_watched_time_rating_8,feat_min_watched_time_rating_9,feat_max_watched_time_rating_9,feat_min_watched_time_rating_10,feat_max_watched_time_rating_10
0,183700,4602,4602,7681,7681,7595,7595


Результат совпал с ожидаемым

Теперь проверим пользователя с двумя оценками:

In [89]:
example_user2 = (
    user_rating_counts
    .filter(F.col("ratings_count") == 2)
    .select("user_uid")
    .first()["user_uid"]
)

print("user:", example_user2)

user: 478224


Посмотрим его исходные просмотры и оценки:

In [90]:
views_ratings.filter(F.col("user_uid") == example_user2).orderBy("rating", "watched_time").show()

+--------+-----------+------------+------+-------------------+
|user_uid|element_uid|watched_time|rating|                 ts|
+--------+-----------+------------+------+-------------------+
|  478224|       7428|        5863|     8|4.302605585201259E7|
|  478224|       4929|        6174|     8|4.302746688527205E7|
|  478224|        344|        6181|     9|4.180829333877019E7|
+--------+-----------+------------+------+-------------------+



То есть у него min по оценке 8 должен быть 5863, max 6174. По оценке 9 min=max=6181, по оценке 10 min=max=NULL

Его итоговые признаки:

In [91]:
user_features.filter(F.col("user_uid") == example_user2).toPandas()

,user_uid,feat_min_watched_time_rating_8,feat_max_watched_time_rating_8,feat_min_watched_time_rating_9,feat_max_watched_time_rating_9,feat_min_watched_time_rating_10,feat_max_watched_time_rating_10
0,478224,5863,6174,6181,6181,NaN,NaN


Результат совпал с ожидаемым

Теперь проверим пользователя с одной оценкой:

In [92]:
example_user1 = (
    user_rating_counts
    .filter(F.col("ratings_count") == 1)
    .select("user_uid")
    .first()["user_uid"]
)

print("user:", example_user1)

user: 549225


Посмотрим его исходные просмотры и оценки:

In [93]:
views_ratings.filter(F.col("user_uid") == example_user1).orderBy("rating", "watched_time").show()

+--------+-----------+------------+------+--------------------+
|user_uid|element_uid|watched_time|rating|                  ts|
+--------+-----------+------------+------+--------------------+
|  549225|        747|       11389|    10|4.1986658598795995E7|
|  549225|       7609|       13999|    10| 4.237951905218316E7|
+--------+-----------+------------+------+--------------------+



То есть у него min по оценке 10 должен быть 11389, max 13999. По оценкам 9 и 10 min=max=NULL

Его итоговые признаки:

In [94]:
user_features.filter(F.col("user_uid") == example_user1).toPandas()

,user_uid,feat_min_watched_time_rating_8,feat_max_watched_time_rating_8,feat_min_watched_time_rating_9,feat_max_watched_time_rating_9,feat_min_watched_time_rating_10,feat_max_watched_time_rating_10
0,549225,NaN,NaN,NaN,NaN,11389,13999


Результат совпал с ожидаемым

Осталовь проверить пользователя без оценок:

In [98]:
users_without_ratings = users_df.join(views_ratings.select("user_uid").distinct(), "user_uid", "left_anti")

In [99]:
example_user0 = users_without_ratings.first()["user_uid"]

print("user:", example_user0)

user: 528372


У него все 6 признаков должны быть NULL

In [100]:
user_features.filter(F.col("user_uid") == example_user0).toPandas()

,user_uid,feat_min_watched_time_rating_8,feat_max_watched_time_rating_8,feat_min_watched_time_rating_9,feat_max_watched_time_rating_9,feat_min_watched_time_rating_10,feat_max_watched_time_rating_10
0,528372,NaN,NaN,NaN,NaN,NaN,NaN


Результат совпал с ожидаемым